# CCORE GLB-3D-Coder 4B — Three.js game assets on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/GLB-3D-Coder-4B-threejs-assets.ipynb)

[`ccore/Qwen3.5-4B-GLB-3D-Coder`](https://huggingface.co/ccore/Qwen3.5-4B-GLB-3D-Coder) is an **Apache-2.0** 4B coding finetune of [`Qwen/Qwen3.5-4B`](https://huggingface.co/Qwen/Qwen3.5-4B). Describe a low-poly character, prop, building, or environment in English, and the model writes a **complete standalone Three.js HTML page** with a preview and an **Export GLB** button. Open the page, inspect the asset, and export a `.glb` for Godot or another editor that supports GLB.

**Why this model.** It is specialized for **procedural low-poly game assets**, not general chat. The Hub ships **Q4_K_M GGUF (2.7 GB)**, **Q8_0 GGUF (4.5 GB)** — the quant used for the recorded evaluation — and the original safetensors (five shards, about **17.49 GB**, FP32 text + BF16 vision). This notebook loads **Q4_K_M** with a **prebuilt CUDA `llama-cpp-python` wheel**. Do **not** compile llama.cpp from source on Colab; source builds disconnect the free T4 runtime. The 17.49 GB original checkpoint does not fit a 15 GB T4.

See [Use cases](https://github.com/donvito/notebooks/blob/main/docs/GLB-3D-Coder-4B-threejs-assets.md) for when this notebook is a good fit and when it is not.

**Licence:** Apache-2.0 (public, not gated). Cancel any Colab `HF_TOKEN` prompt; a token is optional for higher Hub rate limits.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. A free T4 has ~15 GB of VRAM. This notebook uses **`n_ctx=32768`**. The card says an **8K output limit is too restrictive** (responses average around **20K tokens**, some longer; training examples reach **72K**). The full **262,144**-token window from `config.json` is not allocated. Sampler is the card's: temperature **0.8**, top_k **40**, top_p **0.95**, min_p **0.0**, repeat penalty **1.0**, **thinking enabled**. GGUF architecture is **`qwen35`** (llama.cpp from b7990). This notebook pins **`llama-cpp-python` 0.3.37** (CUDA 12.1 wheel).

**Not in this notebook:** the original safetensors / Transformers path, image inputs (the optional `mmproj` projector is not loaded), the RIFTWARD arena Space, or a rerun of any unpublished eval.

**Links:** [model card](https://huggingface.co/ccore/Qwen3.5-4B-GLB-3D-Coder) · [training dataset](https://huggingface.co/datasets/ccore/ThreeJS-3D-GLB-Assets) · [announcement](https://x.com/alexdelrosso/status/2123780570526142700) · [use cases](https://github.com/donvito/notebooks/blob/main/docs/GLB-3D-Coder-4B-threejs-assets.md)


## 0. Check the GPU

Confirm CUDA is available, print the device name and free VRAM. Need a T4 (or any CUDA GPU). A restart after the install cell drops this state; section 2 prints it again before load.


In [ ]:
import os
import shutil

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

import torch

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch", torch.__version__, "| cuda", torch.version.cuda)

if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info()
    major, minor = torch.cuda.get_device_capability()
    print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
    print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
    if major < 8:
        print("No native bfloat16 / FlashAttention-2 on this GPU; GGUF load keeps flash_attn=False.")
else:
    print("GPU name: none")
    print("VRAM free: n/a")
    print("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")


## 1. Install the CUDA `llama-cpp-python` wheel

Remove any CPU build and install the **pinned 0.3.37 CUDA 12.1** wheel from the project's GitHub releases. A direct wheel URL stops pip from grabbing the CPU package on PyPI. CUDA 12.1 binaries work with the newer driver on a Colab T4.

`qwen35` landed in llama.cpp **b7990**. **0.3.17** is the first `llama-cpp-python` with Qwen 3.5 support; **0.3.37** vendors llama.cpp `@a956a852`. Do **not** use an older 0.3.16-or-earlier wheel, and do **not** compile llama.cpp / `llama-cpp-python` from source here — that compile disconnects the free Colab runtime.

Current free Colab images (Python 3.12/3.13) often ship the NVIDIA driver but not the CUDA 12 user-space libraries the wheel links (`libcudart.so.12`, `libcublas.so.12`), so the install also pulls `nvidia-cuda-runtime-cu12` and `nvidia-cublas-cu12`. Use `%pip` (not `python -I -m pip`) so Colab's site-packages stay on the path.

**Do not import `llama_cpp` in this cell.** After reconnect, skip this cell and continue from section 2.


In [ ]:
import importlib.metadata as md
import os
import signal
import sys
from pathlib import Path

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

INSTALL_TAG = "llama-cpp-python==0.3.37-cu121-qwen35-glb-3d-coder"
MARKER = (
    Path("/content/.glb_3d_coder_nb_install_ok")
    if Path("/content").is_dir()
    else Path(".glb_3d_coder_nb_install_ok")
)
already_restarted = MARKER.exists() and MARKER.read_text().strip() == INSTALL_TAG

print("python", sys.version.split()[0])
try:
    import torch

    print("torch (keep Colab's wheel)", torch.__version__, "| cuda", torch.version.cuda)
except ImportError:
    print("torch not imported yet")

# Public weights: skip Colab userdata.get("HF_TOKEN") popups.
try:
    import huggingface_hub.utils._auth as _hf_auth

    _hf_auth._IS_GOOGLE_COLAB_CHECKED = True
    _hf_auth._GOOGLE_COLAB_SECRET = None
except Exception:
    pass

WHEEL = (
    "https://github.com/abetlen/llama-cpp-python/releases/download/"
    "v0.3.37-cu121/llama_cpp_python-0.3.37-py3-none-manylinux_2_35_x86_64.whl"
)
print("llama-cpp-python CUDA wheel:", WHEEL.rsplit("/", 2)[-2])

get_ipython().run_line_magic("pip", "uninstall -qqq -y llama-cpp-python")
# Free Colab T4 often has a driver but no CUDA 12 user-space libs.
get_ipython().run_line_magic(
    "pip", "install -qqq --no-cache-dir nvidia-cuda-runtime-cu12 nvidia-cublas-cu12"
)
get_ipython().run_line_magic("pip", f"install -qqq --no-cache-dir {WHEEL}")
get_ipython().run_line_magic("pip", 'install -qqq --no-cache-dir "huggingface_hub>=0.23.0"')

print()
print("llama-cpp-python", md.version("llama-cpp-python"))
print("huggingface_hub", md.version("huggingface_hub"))

MARKER.write_text(INSTALL_TAG)
if already_restarted:
    print("Install marker already set; not restarting again. Continue from section 2.")
else:
    print()
    print("=" * 72)
    print("Restarting the session so the CUDA llama_cpp native library is imported.")
    print("After reconnect: skip this cell and run from section 2.")
    print("A 'session crashed' or 'Operation cancelled' message here is the restart,")
    print("not a failed install — pip already finished (versions printed above).")
    print("=" * 72)
    os.kill(os.getpid(), signal.SIGKILL)


> The install cell **restarts the session for you** so the CUDA `llama_cpp` wheel replaces any CPU build already imported. After reconnect, skip it and continue from section 2.
> A "session crashed" message after install is the restart, not a failed pip. If you restarted by hand instead: *Runtime > Restart session*, then this next cell.


## 2. Runtime check after restart

Confirm CUDA, the GPU name, free VRAM, and that `llama-cpp-python` is **0.3.37+** (Qwen 3.5 / `qwen35` architecture). T4 (`sm_75`) has no FlashAttention-2 — load keeps `flash_attn=False`.


In [ ]:
import ctypes
import glob
import importlib
import os
import shutil
import subprocess

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

try:
    import huggingface_hub.utils._auth as _hf_auth

    _hf_auth._IS_GOOGLE_COLAB_CHECKED = True
    _hf_auth._GOOGLE_COLAB_SECRET = None
except Exception:
    pass

import torch
from packaging.version import Version


def _preload_cuda12_runtime() -> None:
    """Make CUDA 12 runtime/cublas visible before importing llama_cpp.

    The cu121 wheel CDLLs libggml-cuda, which needs libcudart.so.12 and
    libcublas.so.12. Those live in the nvidia-*-cu12 pip packages on current
    free Colab, not on the default linker path.
    """
    sos = []
    for pkg_name in ("nvidia.cuda_runtime", "nvidia.nvjitlink", "nvidia.cublas"):
        try:
            pkg = importlib.import_module(pkg_name)
        except ImportError:
            continue
        sos.extend(glob.glob(os.path.join(pkg.__path__[0], "lib", "*.so*")))
    for so in sos + sos:
        try:
            ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
        except OSError:
            continue


PEAK_VRAM_MIB = 0.0


def print_vram(label=""):
    # llama.cpp allocates CUDA memory outside PyTorch. nvidia-smi is the measured counter.
    global PEAK_VRAM_MIB
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.free,memory.total", "--format=csv,noheader"],
        text=True,
    ).strip()
    used = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
        text=True,
    ).strip()
    try:
        PEAK_VRAM_MIB = max(PEAK_VRAM_MIB, float(used))
    except ValueError:
        pass
    suffix = f" ({label})" if label else ""
    print(f"nvidia-smi used/free/total{suffix}: {smi}")


_preload_cuda12_runtime()
import llama_cpp

assert Version(llama_cpp.__version__) >= Version("0.3.17"), (
    f"llama-cpp-python {llama_cpp.__version__} is loaded; Qwen3.5 GGUF needs 0.3.17+ "
    "(qwen35 architecture). Run the install cell, then Runtime > Restart session."
)
assert Version(llama_cpp.__version__) >= Version("0.3.37"), (
    f"llama-cpp-python {llama_cpp.__version__} is loaded; this notebook pins 0.3.37 "
    "(CUDA 12.1 wheel). Run the install cell, then Runtime > Restart session."
)

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch", torch.__version__, "| cuda", torch.version.cuda)
print("llama-cpp-python", llama_cpp.__version__)
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

major, minor = torch.cuda.get_device_capability()
free_b, total_b = torch.cuda.mem_get_info()
print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")
print()
print("=== T4-first notes ===")
print("- Do NOT compile llama.cpp. Use the 0.3.37 cu121 wheel (previous cell).")
print("- GGUF architecture must be qwen35 (printed after load).")
print("- Default quant: Q4_K_M (2.7 GB). Eval quant on the card: Q8_0 (4.5 GB).")
print("- Original safetensors are ~17.49 GB (FP32 text + BF16 vision) and do not fit a T4.")
print("- Card sampler: temperature=0.8, top_k=40, top_p=0.95, min_p=0.0, repeat_penalty=1.0.")
print("- Thinking stays ON. Card: do not disable, truncate, or budget thinking tokens.")
print("- n_ctx=32768. Card: 8K output is too small; responses average ~20K tokens.")
print("- flash_attn=False on sm_75 (T4 has no FlashAttention-2).")
print_vram("before load")


## 3. Load the Q4_K_M GGUF on the T4

`Llama.from_pretrained` downloads `Qwen3.5-4B-GLB-3D-Coder-Q4_K_M.gguf` (**2.7 GB**) from [`ccore/Qwen3.5-4B-GLB-3D-Coder`](https://huggingface.co/ccore/Qwen3.5-4B-GLB-3D-Coder). `n_gpu_layers=-1` offloads every layer. `n_ctx=32768` is a Colab-practical window: the card says 8K is too small for complete HTML, while the native 262K context is not allocated. `flash_attn=False` is required on sm_75.

To try the **Q8_0** file used for the recorded evaluation (4.5 GB), set `FILENAME` to `Qwen3.5-4B-GLB-3D-Coder-Q8_0.gguf`. After load, the cell prints `general.architecture` and stops if it is not `qwen35`.


In [ ]:
from llama_cpp import Llama

REPO_ID = "ccore/Qwen3.5-4B-GLB-3D-Coder"
FILENAME = "Qwen3.5-4B-GLB-3D-Coder-Q4_K_M.gguf"
# Eval quant on the card (~4.5 GB):
# FILENAME = "Qwen3.5-4B-GLB-3D-Coder-Q8_0.gguf"
N_CTX = 32768
# Card sampling settings (generation_config.json / README).
SAMPLER = dict(temperature=0.8, top_k=40, top_p=0.95, min_p=0.0, repeat_penalty=1.0)
STOP = ["<|im_end|>", "<|endoftext|>"]
# Card system prompt (Quick start).
SYSTEM_PROMPT = (
    "Create game assets for the Godot editor, which uses GLB files. "
    "Create whatever the user requests as a good-quality low-poly asset using only "
    "Three.js in one complete standalone HTML file. The HTML must show the asset and "
    "include an Export GLB button that downloads a working GLB. If the requested "
    "asset needs animation, add it and include the animation in the exported GLB; "
    "otherwise make it static."
)

llm = Llama.from_pretrained(
    repo_id=REPO_ID,
    filename=FILENAME,
    n_gpu_layers=-1,
    n_ctx=N_CTX,
    flash_attn=False,
    verbose=True,
)

meta = dict(llm.metadata or {})
arch = meta.get("general.architecture")
print()
print("loaded", FILENAME)
print("GGUF architecture:", arch)
print("n_ctx", N_CTX, "| n_gpu_layers -1 | flash_attn False")
print("sampler", SAMPLER)
print("thinking: enabled (card default)")
if arch != "qwen35":
    raise SystemExit(
        f"Expected GGUF architecture 'qwen35', got {arch!r}. "
        "The installed llama-cpp-python wheel does not support this Qwen3.5 file. "
        "Re-run the 0.3.37 cu121 install cell (do not compile from source)."
    )
print_vram("after load")


## 4. Generate, save, and preview HTML

The card: send an English asset request, **keep thinking enabled**, allow enough output tokens for the complete HTML (`-1` / unlimited where supported), then save the page as `.html` and open it. This helper:

1. Calls `create_chat_completion` with the card system prompt, sampler, and `enable_thinking=True`.
2. Streams tokens so a ~20K-token page does not look hung.
3. Strips the `<think>` block, pulls the HTML document out of markdown fences if present, and writes it to disk.
4. Renders it inline with an iframe (`srcdoc` + `sandbox` so Three.js and **Export GLB** can run). If the Colab output iframe blocks a CDN script, download the `.html` and open it locally.

Generation of a complete page can take several minutes on a T4. The card: some requests need a retry or a small code correction; materials, lighting, and animations can differ from the browser preview.


In [ ]:
import html as html_lib
import re
import time
from pathlib import Path

from IPython.display import HTML, display

ROOT = Path("/content") if Path("/content").is_dir() else Path(".")
OUT_DIR = ROOT / "glb_assets"
OUT_DIR.mkdir(parents=True, exist_ok=True)


def qwen35_thinking_prompt(system_prompt: str, user_prompt: str) -> str:
    """Qwen3.5 chat template with thinking ON (opens an unclosed <think> block)."""
    return (
        f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
        f"<|im_start|>user\n{user_prompt}<|im_end|>\n"
        "<|im_start|>assistant\n<think>\n"
    )


def strip_think(text: str) -> str:
    text = text.strip()
    if "</think>" in text:
        text = text.split("</think>", 1)[-1].strip()
    elif text.startswith("<think>"):
        # Generation stopped during thinking.
        return ""
    return text


def extract_html(raw: str) -> str:
    text = strip_think(raw)
    fence = re.search(r"```(?:html)?\s*([\s\S]*?)```", text, re.IGNORECASE)
    if fence:
        candidate = fence.group(1).strip()
        if re.search(r"<!doctype\s+html|<html", candidate, re.IGNORECASE):
            text = candidate
    lower = text.lower()
    start = lower.find("<!doctype")
    if start < 0:
        start = lower.find("<html")
    if start < 0:
        return text.strip()
    end = lower.rfind("</html>")
    if end >= 0:
        return text[start : end + len("</html>")].strip()
    return text[start:].strip()


def preview_html(html_text: str, height: int = 540) -> None:
    escaped = html_lib.escape(html_text)
    display(
        HTML(
            f'<iframe srcdoc="{escaped}" width="100%" height="{height}" '
            'sandbox="allow-scripts allow-downloads allow-modals allow-popups allow-same-origin" '
            'style="border:1px solid #ccc;border-radius:8px;background:#111"></iframe>'
        )
    )


def generate_asset(user_prompt: str, outfile: Path, max_tokens: int = -1) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    print("user prompt:", user_prompt)
    print("max_tokens:", max_tokens, "| n_ctx", N_CTX)
    t0 = time.time()
    pieces = []
    in_html = False
    since_report = 0

    def _on_piece(piece: str) -> None:
        nonlocal in_html, since_report
        if not piece:
            return
        pieces.append(piece)
        joined_tail = "".join(pieces[-8:])
        if (not in_html) and ("</think>" in joined_tail or "<!doctype" in joined_tail.lower() or "<html" in joined_tail.lower()):
            in_html = True
            print("\n--- HTML ---", flush=True)
        if in_html:
            since_report += len(piece)
            if since_report >= 2000:
                print(f"  … {sum(len(p) for p in pieces)} chars", flush=True)
                since_report = 0
        else:
            print(piece, end="", flush=True)

    def _consume_stream(stream) -> None:
        for chunk in stream:
            choice = (chunk.get("choices") or [{}])[0]
            delta = choice.get("delta") or {}
            _on_piece(delta.get("reasoning_content") or "")
            _on_piece(delta.get("content") or choice.get("text") or "")

    print("--- think ---")
    try:
        stream = llm.create_chat_completion(
            messages=messages,
            max_tokens=max_tokens,
            stop=STOP,
            stream=True,
            chat_template_kwargs={"enable_thinking": True},
            **SAMPLER,
        )
        _consume_stream(stream)
    except TypeError:
        # Older kwargs (min_p / chat_template_kwargs) or no streaming chat path.
        sampler = {k: v for k, v in SAMPLER.items() if k != "min_p"}
        try:
            stream = llm.create_chat_completion(
                messages=messages,
                max_tokens=max_tokens,
                stop=STOP,
                stream=True,
                **sampler,
            )
            _consume_stream(stream)
        except TypeError:
            out = llm(
                qwen35_thinking_prompt(SYSTEM_PROMPT, user_prompt),
                max_tokens=max_tokens if max_tokens > 0 else N_CTX - 512,
                stop=STOP,
                stream=True,
                **sampler,
            )
            for chunk in out:
                _on_piece((chunk.get("choices") or [{}])[0].get("text") or "")
    print()
    raw = "".join(pieces)
    elapsed = time.time() - t0
    print(f"--- {len(raw)} chars in {elapsed:.1f}s ---")
    html_text = extract_html(raw)
    if not html_text:
        print("warning: no HTML after the think block (generation may have stopped while thinking).")
        html_text = raw
    elif not re.search(r"</html>", html_text, re.IGNORECASE):
        print("warning: HTML looks incomplete (no </html>). Card: allow the model to finish; 8K is too small.")
    outfile.write_text(html_text, encoding="utf-8")
    print("wrote", outfile, f"({outfile.stat().st_size} bytes)")
    print_vram("after generate")
    preview_html(html_text)
    return html_text


## 5. Demo 1 — low-poly treasure chest

Short English request, as the card suggests to start. Static prop (no animation asked). Save `treasure-chest.html` and preview it inline. Click **Export GLB** in the iframe if the page rendered.


In [ ]:
chest_html = generate_asset(
    "Create a low-poly wooden treasure chest with iron bands and a gold latch.",
    OUT_DIR / "treasure-chest.html",
)
print("preview above; file:", OUT_DIR / "treasure-chest.html")


## 6. Demo 2 — small house

A short architecture prompt (the card's domains include modular architecture / homestead). Static. Save `small-house.html`.


In [ ]:
house_html = generate_asset(
    "Create a small low-poly house with a red roof, a wooden door, and two windows.",
    OUT_DIR / "small-house.html",
)
print("preview above; file:", OUT_DIR / "small-house.html")
print()
print("saved assets:")
for path in sorted(OUT_DIR.glob("*.html")):
    print(f"  {path}  {path.stat().st_size} bytes")


## 7. Caveats and cleanup

**What this notebook is for.** The card's job on a free T4: turn an English asset prompt into one complete Three.js HTML file with a preview and an Export GLB button, then inspect it in Colab.

**What it is not.** General-purpose coding, photoreal 3D, or a mesh diffusion model. The card: some requests need a retry or a small code correction; inspect the exported asset in your editor, since materials, lighting, and animations can differ from the browser preview. Image-conditioned generation is not demoed (optional `mmproj-Qwen3.5-4B-BF16.gguf` is 644 MiB and is only for image inputs in a compatible multimodal GGUF runtime).

**Quant.** Default is **Q4_K_M (2.7 GB)** so a 32K context fits a T4 with room to spare. **Q8_0 (4.5 GB)** is the quant used for the recorded evaluation. Original safetensors (~17.49 GB) are not loaded.

**Output length.** Card: set the output limit to unlimited (`-1`); an 8K cap can stop during thinking or before the HTML is complete. This notebook uses `max_tokens=-1` inside `n_ctx=32768`. Responses average around 20K tokens and can take several minutes on a T4.

**Thinking.** Card: keep thinking enabled, with no effort limit or separate thinking-token budget. The author has not validated quality with thinking disabled.

**llama.cpp.** Architecture string is **`qwen35`**. The 0.3.37 CUDA wheel vendors a llama.cpp new enough to load it. Do not compile from source on Colab.

**Peak VRAM.** `print_vram` reports `nvidia-smi` used/free/total because llama.cpp does not go through `torch.cuda.memory_allocated`. Q4_K_M is 2.7 GB on disk; used VRAM after load is weights + KV cache + CUDA workspace.

**Related notebooks:** [FrogNano-4B coding agent](FrogNano-4B-coding-agent.ipynb) (Leaf SWE loop, not 3D assets), [Mellum2.1 12B-A2.5B Thinking coding agent](Mellum2.1-12B-A2.5B-Thinking-coding-agent.ipynb) (unified-diff coding agent), [Shot Reader 8B video shot to 3D](shot-reader-8b-video-shot-to-3d-scene.ipynb) (video → JSON shot sheet, not GLB HTML).

Free the GPU before you open another notebook:


In [ ]:
import gc

print_vram("before cleanup")
try:
    used = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
        text=True,
    ).strip()
    session_used_gb = float(used) / 1024
    peak_gb = PEAK_VRAM_MIB / 1024
    print(f"nvidia-smi used before free: {session_used_gb:.2f} GB")
    print(f"peak VRAM this session (max nvidia-smi used): {peak_gb:.2f} GB")
except Exception as exc:
    print("could not read nvidia-smi used:", exc)

del llm
gc.collect()
print_vram("after cleanup")
